# Train the 2.5D tumor segmentation UNet (TCGA-LGG)

**Before running:** `Runtime → Change runtime type → T4 GPU`.

1. Put the Kaggle *LGG MRI Segmentation* dataset in your Drive as `kaggle_3m.zip` (or as an extracted `kaggle_3m` folder).
2. Run the cells in order. Training writes to `runs/<name>/` in Colab and the last cell copies it to your Drive.

The protocol is grouped 5-fold CV over patients, with the probability threshold chosen on each fold's validation split only — never on test — plus per-patient Dice3D and volume error.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
!pip install -q segmentation-models-pytorch==0.6.0 opencv-python-headless

In [ ]:
import os

# --- code ---
if not os.path.isdir("/content/repo"):
    !git clone -q https://github.com/arunprashath06/Brain-Tumor-Segmentation.git /content/repo

# --- dataset: edit these two paths if yours differ ---
DATA_ZIP = "/content/drive/MyDrive/kaggle_3m.zip"
DATA_DIR = "/content/data"

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(DATA_DIR, exist_ok=True)
if not any(name.startswith("TCGA_") for root, _, _ in os.walk(DATA_DIR) for name in root.split("/")[-1:]):
    if os.path.exists(DATA_ZIP):
        !unzip -q -o "{DATA_ZIP}" -d /content/data
    else:
        print("Dataset not found. Upload it to", DATA_ZIP)

!find /content/data -maxdepth 3 -type d -name 'TCGA_*' | head -3
!find /content/data -maxdepth 3 -type d -name 'TCGA_*' | wc -l

In [ ]:
# Quick sanity check: one fold, one epoch, a few patients (~2 minutes on a T4)
!cd /content/repo && python train.py --data /content/data --out /content/repo/runs/smoke \
  --folds 1 --epochs 1 --limit-patients 8 --size 256 --batch 4 --max-batches 20

In [ ]:
# Full run: 5 folds x 25 epochs. Roughly 2.5-4 hours on a T4.
# Lower --epochs to 15 for a first pass, and keep --batch 8 (drop to 4 if you hit a CUDA OOM).
!cd /content/repo && python train.py --data /content/data --out /content/repo/runs/r34_25d \
  --encoder resnet34 --size 384 --epochs 25 --folds 5 --batch 8 --workers 2

In [ ]:
import json, shutil, os

shutil.copytree("/content/repo/runs", "/content/drive/MyDrive/brain_tumor_runs", dirs_exist_ok=True)
print("saved to Drive: MyDrive/brain_tumor_runs")
print(json.dumps(json.load(open("/content/repo/runs/r34_25d/results.json")), indent=2))

## After training

Copy `runs/r34_25d/` back into the project folder and the Streamlit app picks it up automatically (it discovers any `*/<run>/config.json` next to `best_fold*.pt`). The run's `results.json` carries the honest out-of-fold numbers: slice Dice, Dice3D and volume error.